# Notebook 01 — Preprocessing & Save Artifacts

Loads **Loan_Prediction_Realistic.csv**, engineers features, scales, applies SMOTE,
and saves everything to `../outputs/` for the next notebooks.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd, numpy as np
from src.preprocessing import load_data, handle_missing_values, split_and_scale, apply_smote, save_preprocessed
from src.feature_engineering import create_features

In [2]:
df = load_data('../data/Loan_Prediction_Realistic.csv')
df.head()

Dataset loaded: 12367 rows x 8 columns
  Using realistic dataset with injected noise and label uncertainty


,age,income,assets,credit_score,debt_to_income_ratio,existing_loan,criminal_record,loan_approved
0,53.0,56450.0,838996.0,716.000000,0.440000,1,0,0
1,65.0,139785.0,583848.0,850.000000,0.360000,1,1,0
2,59.0,39758.0,505584.0,765.000000,0.209554,0,1,1
3,65.0,116830.0,175536.0,834.000000,0.370000,1,1,0
4,26.0,81319.0,182040.0,691.466118,0.180000,0,0,0


In [3]:
# Handle any missing values
df = handle_missing_values(df)
print('Missing after fix:', df.isnull().sum().sum())

Missing after fix: 0


In [4]:
# Feature engineering
df_eng, new_feats = create_features(df)
print(f'Columns: {list(df_eng.columns)}')

Created 7 engineered features
  Removed: financial_health (redundant), risk_flags (no new info)
  Added: payment_capacity, credit_utilization_proxy
Columns: ['age', 'income', 'assets', 'credit_score', 'debt_to_income_ratio', 'existing_loan', 'criminal_record', 'loan_approved', 'credit_tier', 'high_debt', 'asset_income_ratio', 'age_group', 'income_per_age', 'payment_capacity', 'credit_utilization_proxy']


In [5]:
# Separate X / y
TARGET = 'loan_approved'
X = df_eng.drop(TARGET, axis=1)
y = df_eng[TARGET]
print(f'X: {X.shape}   y distribution:')
print(y.value_counts())

X: (12367, 14)   y distribution:
loan_approved
0    10185
1     2182
Name: count, dtype: int64


In [6]:
# Split + scale
X_train_sc, X_test_sc, y_train, y_test, scaler = split_and_scale(X, y)

Training set: 9893 samples
Test set:     2474 samples


In [7]:
# SMOTE on training data only
X_train_bal, y_train_bal = apply_smote(X_train_sc, y_train)

Before SMOTE: 8148 rejected, 1745 approved
After  BorderlineSMOTE: 8148 rejected, 8148 approved


In [8]:
# Save
save_preprocessed(
    X_train_bal, X_test_sc, y_train_bal, y_test, scaler,
    X_original=X, y_original=y, output_dir='../outputs'
)
print('\nDone.  Proceed to 02_Model_Building.')

All artefacts saved to ../outputs/

Done.  Proceed to 02_Model_Building.
